## 🎯 Learning Objectives
* Understand the core principles and advantages of Direct Preference Optimization (DPO) as an alternative to traditional Reinforcement Learning from Human Feedback (RLHF) methods.
* Identify the key components of the DPO loss function and how it directly optimizes a policy based on human preferences.
* Implement a simplified DPO training loop to grasp the practical application of the algorithm.
* Evaluate the trade-offs, typical use cases, and performance characteristics of DPO in aligning large language models.


## DPO: Direct Preference Optimization – A Simpler Path to Alignment

In the journey of aligning large language models (LLMs) with human values and instructions, Reinforcement Learning from Human Feedback (RLHF) has been a cornerstone. However, RLHF, particularly methods like Proximal Policy Optimization (PPO), can be complex, computationally intensive, and prone to instability due to the intricate interplay between a reward model, an actor policy, and a value function. Enter **Direct Preference Optimization (DPO)**, a groundbreaking technique that offers a significantly simpler and more stable alternative.

### The Core Idea: Direct Alignment

Imagine you're teaching a student to write better essays. With traditional RLHF, you'd first hire a judge (the reward model) to score essays, then train the student (the policy) to write essays that get high scores from the judge. This indirect approach can be tricky: the judge might be inconsistent, and the student might learn to game the judge rather than truly improve.

DPO takes a more direct route. Instead of training a separate judge, DPO directly teaches the student by showing them pairs of essays and saying, "This essay (chosen) is better than that essay (rejected) for this prompt." The student then learns to adjust their writing style to produce essays more like the 'chosen' ones and less like the 'rejected' ones, relative to their initial writing style. There's no intermediate reward model; the preference signal directly informs the policy update.

### How DPO Works: A Step-by-Step Breakdown

1.  **Data Collection**: The process begins with collecting human preference data. For a given prompt, humans are presented with two or more model responses and asked to choose which one they prefer. This results in pairs of (prompt, chosen_response, rejected_response).

2.  **Reference Model (SFT Model)**: A supervised fine-tuned (SFT) model is trained on high-quality instruction-following data. This model serves as our initial policy and a 'reference' point during DPO training. It represents the model's behavior *before* preference alignment.

3.  **The DPO Loss Function**: This is where the magic happens. DPO formulates a loss function that directly optimizes the policy to maximize the likelihood of chosen responses and minimize the likelihood of rejected responses, *relative to the reference model*. The loss function is derived from the Bradley-Terry model of preferences and incorporates a KL divergence penalty implicitly, preventing the policy from drifting too far from the reference model.

    Mathematically, for a given prompt `x`, chosen response `y_w` (winner), and rejected response `y_l` (loser), the DPO loss for a policy `pi` and reference policy `pi_ref` is typically:

    $$ L_{DPO}(\pi) = - \mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma \left( \beta \log \frac{\pi(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right] $$

    Where:
    *   $\sigma$ is the sigmoid function.
    *   $\beta$ is a hyperparameter that controls the strength of the KL divergence penalty (how much the policy is allowed to deviate from the reference model).
    *   $\frac{\pi(y|x)}{\pi_{ref}(y|x)}$ is the policy's likelihood ratio for a response `y` given prompt `x`, relative to the reference model.

    In essence, DPO tries to make the log-probability ratio of the chosen response to the reference model *higher* than the log-probability ratio of the rejected response to the reference model. The sigmoid function ensures the loss is well-behaved.

4.  **Optimization**: The DPO loss is then used to update the parameters of the policy model using standard gradient descent optimization (e.g., AdamW). This process is much more stable than PPO because it avoids the complexities of reward model training, sampling from the policy, and the on-policy nature of RL algorithms.

### Advantages of DPO

*   **Simplicity**: No need to train a separate reward model. The preference data directly informs the policy. This significantly reduces the number of components and hyperparameters.
*   **Stability**: DPO training is generally more stable and less sensitive to hyperparameters compared to PPO, which often suffers from high variance and convergence issues.
*   **Computational Efficiency**: Eliminates the need for reward model inference during training and complex sampling strategies, leading to faster training times.
*   **Performance**: Empirically, DPO has been shown to achieve comparable or even superior performance to PPO in aligning LLMs, often producing models that are preferred by human evaluators.

By simplifying the alignment pipeline, DPO has become a go-to method for fine-tuning LLMs in 2026, enabling more accessible and robust model alignment.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from accelerate import Accelerator

# --- 1. Configuration and Setup ---
# Using a small, pre-trained model for demonstration purposes.
# In a real-world scenario, this would be a large LLM.
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add a padding token if it doesn't exist (common for some models)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'}) 

# We'll simulate a policy model and a reference model.
# For simplicity, we'll use a sequence classification model and adapt it.
# In reality, these would be generative LLMs.
class ToyGenerativeModel(nn.Module):
    def __init__(self, base_model_name, num_labels=2):
        super().__init__()
        self.base_model = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=num_labels)
        # Resize token embeddings if we added a new token
        self.base_model.resize_token_embeddings(len(tokenizer))

    def forward(self, input_ids, attention_mask=None, labels=None):
        # For DPO, we need log-probabilities of generated sequences.
        # Here, we'll simplify: imagine the 'logits' are scores for two possible responses.
        # We'll use the classification head to simulate a 'preference score' for a given response.
        # This is a *highly simplified* abstraction for demonstration.
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        # We'll treat the logits as log-probabilities for simplicity in this toy example.
        # In a real LLM, you'd get log-probs for each token in the sequence.
        return outputs.logits

# Initialize policy and reference models
policy_model = ToyGenerativeModel(model_name)
reference_model = ToyGenerativeModel(model_name)

# Copy initial weights to reference model and freeze it
reference_model.load_state_dict(policy_model.state_dict())
for param in reference_model.parameters():
    param.requires_grad = False

# Accelerator for mixed precision training and distributed setup
accelerator = Accelerator()

# Move models to device
policy_model, reference_model = accelerator.prepare(policy_model, reference_model)

# --- 2. Synthetic Preference Data ---
# In a real scenario, this data would come from human annotators.
# Each entry: (prompt, chosen_response_text, rejected_response_text)
# For our toy model, we'll simplify: chosen/rejected are just different 'labels' or 'scores'
# We'll simulate by having the model predict a 'preference score' for a given (prompt + response) pair.
# A higher score for chosen, lower for rejected.

# Let's create synthetic data where 'chosen' is associated with label 1 and 'rejected' with label 0
# The model will learn to output a higher logit for label 1 for chosen, and label 0 for rejected.

synthetic_data = [
    {"prompt": "Write a short story about a brave knight.", 
     "chosen": "The knight, Sir Kael, faced the dragon with unwavering courage, his sword gleaming.", 
     "rejected": "A knight was there. He fought a dragon. It was okay."}, 
    {"prompt": "Explain quantum entanglement simply.", 
     "chosen": "Imagine two coins, if one is heads, the other is always tails, no matter the distance.", 
     "rejected": "Quantum entanglement is a phenomenon where particles become interconnected."}, 
    {"prompt": "Suggest a healthy breakfast.", 
     "chosen": "Oatmeal with berries and nuts, rich in fiber and antioxidants.", 
     "rejected": "Bacon and eggs, a classic but high-fat choice."} 
]

# --- 3. DPO Loss Function Implementation ---
def dpo_loss(policy_logits_chosen, policy_logits_rejected, 
             ref_logits_chosen, ref_logits_rejected, beta=0.1):
    
    # Calculate log-probability ratios
    # For our toy model, logits directly represent a 'score' or 'preference'.
    # In a real LLM, these would be log-probs of the *entire sequence*.
    pi_logratios_chosen = policy_logits_chosen
    pi_logratios_rejected = policy_logits_rejected
    
    ref_logratios_chosen = ref_logits_chosen
    ref_logratios_rejected = ref_logits_rejected

    # The DPO objective is based on the difference in log-ratios
    # log(pi(y_w|x)/pi_ref(y_w|x)) - log(pi(y_l|x)/pi_ref(y_l|x))
    # which simplifies to (log pi(y_w|x) - log pi_ref(y_w|x)) - (log pi(y_l|x) - log pi_ref(y_l|x))
    # For our toy model, we're using logits as direct 'scores' for chosen/rejected.
    # So, we'll interpret policy_logits_chosen as log pi(y_w|x) and ref_logits_chosen as log pi_ref(y_w|x)
    
    # The DPO paper's loss is -log(sigmoid(beta * (r_pi - r_ref)))
    # where r_pi = log(pi(y_w|x)) - log(pi(y_l|x))
    # and r_ref = log(pi_ref(y_w|x)) - log(pi_ref(y_l|x))
    
    # Let's adapt this for our simplified logit interpretation:
    # We want policy_logits_chosen to be higher than policy_logits_rejected.
    # The DPO loss encourages: log(pi(chosen)/pi_ref(chosen)) > log(pi(rejected)/pi_ref(rejected))
    # This means: (policy_logits_chosen - ref_logits_chosen) > (policy_logits_rejected - ref_logits_rejected)
    
    # The term inside the sigmoid is: beta * (log_ratio_chosen - log_ratio_rejected)
    # log_ratio_chosen = policy_logits_chosen - ref_logits_chosen
    # log_ratio_rejected = policy_logits_rejected - ref_logits_rejected
    
    # For our classification model, we'll assume policy_logits_chosen is the logit for the 'chosen' class (e.g., label 1)
    # and policy_logits_rejected is the logit for the 'rejected' class (e.g., label 0).
    # This is a simplification. In a true generative model, these would be log-probabilities of sequences.
    
    # Let's assume the model outputs a single logit for 'preference'.
    # We want the logit for the chosen response to be higher than for the rejected response.
    # The DPO loss is derived from the Bradley-Terry model.
    # The term inside the sigmoid is: beta * (log_prob_ratio_chosen - log_prob_ratio_rejected)
    # where log_prob_ratio_chosen = log(pi(y_w|x)) - log(pi_ref(y_w|x))
    
    # For our toy model, we'll make a further simplification:
    # Assume policy_logits_chosen is the log-prob of the chosen sequence under the current policy
    # and policy_logits_rejected is the log-prob of the rejected sequence under the current policy.
    # Similarly for ref_logits.
    
    # Calculate the log-probability ratios for chosen and rejected responses
    log_ratio_chosen = policy_logits_chosen - ref_logits_chosen
    log_ratio_rejected = policy_logits_rejected - ref_logits_rejected
    
    # The DPO objective term
    # We want log_ratio_chosen to be greater than log_ratio_rejected
    # The loss is -log(sigmoid(beta * (log_ratio_chosen - log_ratio_rejected)))
    # This encourages (log_ratio_chosen - log_ratio_rejected) to be large and positive.
    
    dpo_term = beta * (log_ratio_chosen - log_ratio_rejected)
    loss = -torch.nn.functional.logsigmoid(dpo_term)
    
    return loss.mean()

# --- 4. Training Loop ---
optimizer = optim.AdamW(policy_model.parameters(), lr=1e-5)
optimizer = accelerator.prepare(optimizer)

num_epochs = 3
batch_size = 2 # Small batch size for demonstration

print("Starting DPO training...")

for epoch in range(num_epochs):
    total_loss = 0
    for i in range(0, len(synthetic_data), batch_size):
        batch = synthetic_data[i:i+batch_size]
        
        policy_model.train()
        optimizer.zero_grad()
        
        batch_loss = 0
        for item in batch:
            prompt = item["prompt"]
            chosen_response = item["chosen"]
            rejected_response = item["rejected"]
            
            # Tokenize prompt + response pairs
            # For a real generative model, you'd get log-probs of generated sequences.
            # Here, we'll tokenize (prompt + response) and get a 'preference score' from our toy classifier.
            
            # Simulate getting a 'score' for the chosen response
            chosen_input = tokenizer(prompt + " " + chosen_response, return_tensors="pt", 
                                     padding="max_length", truncation=True, max_length=128)
            chosen_input = {k: v.to(accelerator.device) for k, v in chosen_input.items()}
            policy_logits_chosen = policy_model(**chosen_input)[:, 1] # Assuming label 1 is 'preferred'
            ref_logits_chosen = reference_model(**chosen_input)[:, 1]
            
            # Simulate getting a 'score' for the rejected response
            rejected_input = tokenizer(prompt + " " + rejected_response, return_tensors="pt", 
                                       padding="max_length", truncation=True, max_length=128)
            rejected_input = {k: v.to(accelerator.device) for k, v in rejected_input.items()}
            policy_logits_rejected = policy_model(**rejected_input)[:, 1] # Assuming label 1 is 'preferred'
            ref_logits_rejected = reference_model(**rejected_input)[:, 1]
            
            # Calculate DPO loss for this pair
            pair_loss = dpo_loss(policy_logits_chosen, policy_logits_rejected,
                                 ref_logits_chosen, ref_logits_rejected, beta=0.1)
            batch_loss += pair_loss
            
        batch_loss /= len(batch) # Average loss per item in batch
        accelerator.backward(batch_loss)
        optimizer.step()
        
        total_loss += batch_loss.item()
        
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {total_loss / (len(synthetic_data) / batch_size):.4f}")

print("DPO training complete.")

# --- 5. Post-training Evaluation (Simplified) ---
# In a real scenario, you'd evaluate with human preferences or specific metrics.
# Here, we'll just show how the model's 'preference scores' might change.

policy_model.eval()
print("\n--- Post-training Preference Scores (Higher is better) ---")
with torch.no_grad():
    for item in synthetic_data:
        prompt = item["prompt"]
        chosen_response = item["chosen"]
        rejected_response = item["rejected"]

        chosen_input = tokenizer(prompt + " " + chosen_response, return_tensors="pt", 
                                 padding="max_length", truncation=True, max_length=128)
        chosen_input = {k: v.to(accelerator.device) for k, v in chosen_input.items()}
        policy_score_chosen = policy_model(**chosen_input)[:, 1].item()
        ref_score_chosen = reference_model(**chosen_input)[:, 1].item()

        rejected_input = tokenizer(prompt + " " + rejected_response, return_tensors="pt", 
                                   padding="max_length", truncation=True, max_length=128)
        rejected_input = {k: v.to(accelerator.device) for k, v in rejected_input.items()}
        policy_score_rejected = policy_model(**rejected_input)[:, 1].item()
        ref_score_rejected = reference_model(**rejected_input)[:, 1].item()

        print(f"Prompt: {prompt[:50]}...")
        print(f"  Chosen: Policy Score={policy_score_chosen:.4f}, Ref Score={ref_score_chosen:.4f}")
        print(f"  Rejected: Policy Score={policy_score_rejected:.4f}, Ref Score={ref_score_rejected:.4f}")
        print(f"  Policy Diff (Chosen - Rejected): {policy_score_chosen - policy_score_rejected:.4f}")
        print(f"  Ref Diff (Chosen - Rejected): {ref_score_chosen - ref_score_rejected:.4f}")
        print("--------------------------------------------------")

# Expected outcome: policy_score_chosen - policy_score_rejected should increase compared to ref_score_chosen - ref_score_rejected
# or at least policy_score_chosen should increase relative to policy_score_rejected.


### Interpreting the Code Output and DPO's Practicalities

The provided code offers a highly simplified, conceptual illustration of DPO. In a real-world scenario, the `ToyGenerativeModel` would be a full-fledged large language model (LLM) capable of generating text, and the `policy_logits_chosen`/`rejected` would represent the *log-probabilities of the entire generated sequence* under the current policy, not just a single classification logit. However, this toy example effectively demonstrates the core mechanism of the DPO loss function and its application.

#### Interpreting the Output:

*   **Average Loss**: You should observe the `Average Loss` decreasing over epochs. This indicates that the `policy_model` is successfully learning to assign higher relative probabilities to chosen responses compared to rejected ones, aligning itself with the human preferences encoded in the synthetic data.
*   **Post-training Preference Scores**: The most telling part of the output is the comparison of `Policy Diff (Chosen - Rejected)` versus `Ref Diff (Chosen - Rejected)`. After DPO training, you would ideally see:
    *   The `Policy Diff` (the difference between the policy's score for the chosen response and the rejected response) should be *higher* than the `Ref Diff` (the same difference for the reference model).
    *   This signifies that the policy model has learned to *prefer* the chosen responses more strongly and *disprefer* the rejected responses more strongly, relative to its initial state (the reference model).
    *   A positive and increasing `Policy Diff` indicates successful alignment.

#### Performance Trade-offs:

**Advantages:**

*   **Simplicity and Stability**: DPO's primary advantage is its direct optimization approach. By avoiding the explicit reward model and complex sampling of PPO, it leads to a much simpler implementation and significantly more stable training. This translates to fewer hyperparameters to tune and less debugging effort.
*   **Computational Efficiency**: Without the need to train and query a separate reward model during the inner loop of training, DPO is generally more computationally efficient, especially for large models and datasets.
*   **Memory Footprint**: DPO typically requires less GPU memory as it doesn't need to store gradients for a reward model or maintain multiple copies of the policy for on-policy updates.
*   **Strong Empirical Results**: In 2026, DPO has proven to be a highly effective method for aligning LLMs, often matching or exceeding the performance of more complex RLHF methods in terms of human preference scores.

**Disadvantages:**

*   **Data Quality Dependence**: DPO's effectiveness is heavily reliant on the quality and quantity of the human preference data. Poorly annotated or insufficient data can lead to suboptimal alignment.
*   **Reference Model Sensitivity**: The choice of the initial SFT model (which becomes the reference model) is crucial. If the reference model is already very poor, DPO might struggle to make significant improvements.
*   **Exploration Limitations**: As a direct optimization method, DPO might be less exploratory than true RL algorithms. While this is often a benefit for alignment (preventing undesirable drifts), it could be a limitation in scenarios requiring broad exploration of the policy space.

#### Typical Use Cases:

DPO has become a standard technique for:

*   **LLM Alignment**: Fine-tuning LLMs to follow instructions better, generate safer content, adhere to specific stylistic guidelines, or embody desired personality traits.
*   **Personalization**: Adapting generative models to individual user preferences or specific organizational standards.
*   **Safety and Ethics**: Enhancing model safety by training it to avoid generating harmful, biased, or untruthful content based on human feedback.
*   **Domain-Specific Refinement**: Aligning models for specialized domains where specific nuances and preferences are critical (e.g., legal, medical, creative writing).

In the modern AI landscape of 2026, DPO stands out as a powerful, elegant, and practical solution for bringing generative AI models closer to human expectations and values.


### Resources for Further Learning

To deepen your understanding and explore practical implementations of DPO, consider the following resources:

*   **Original DPO Paper**: "Direct Preference Optimization: Your Language Model is Secretly a Reward Model" by Rafael Rafailov, Archit Sharma, Eric Mitchell, Stefano Ermon, Christopher D. Manning, and Jure Leskovec (2023). This paper introduces the theoretical foundations and empirical results of DPO.
    *   [arXiv Link](https://arxiv.org/abs/2305.18290)

*   **Hugging Face `trl` Library**: The `trl` (Transformer Reinforcement Learning) library by Hugging Face provides robust and easy-to-use implementations of DPO, PPO, and other RLHF algorithms. It's the go-to tool for applying these techniques to large transformer models.
    *   [Hugging Face `trl` Documentation](https://huggingface.co/docs/trl/main/en/dpo_trainer)
    *   [DPO Trainer Example](https://huggingface.co/docs/trl/main/en/dpo_trainer)

*   **PyTorch Documentation**: For a deeper dive into the underlying deep learning framework, including optimizers, loss functions, and model architecture.
    *   [PyTorch Official Website](https://pytorch.org/)

*   **Google AI Studio / Gemini API**: While DPO is primarily a fine-tuning technique for existing models, understanding the capabilities and alignment goals of state-of-the-art models like Gemini (accessible via Google AI Studio) provides context for *why* alignment methods like DPO are so crucial.
    *   [Google AI Studio](https://ai.google.dev/)

*   **Accelerate Library**: Hugging Face's `accelerate` library simplifies distributed training and mixed-precision training, which is essential for working with large models and DPO.
    *   [Hugging Face Accelerate Documentation](https://huggingface.co/docs/accelerate/index)

These resources will equip you with both the theoretical knowledge and practical tools to implement and experiment with DPO in your own projects.
